In [1]:
data=open("data/tinyshakespeare.txt").read()
char_list=list(sorted(set(data)))
len(char_list)



65

In [2]:
#tokenizing it
stoi={ch:i for i,ch in enumerate(char_list)}
itos={i:ch for i,ch in enumerate(char_list)}
encode=lambda s:[stoi[c] for c in s]
decode=lambda l:"".join([itos[i] for i in l])




In [3]:
import torch

training_text=data[:int(len(data)*0.9)]
dev_text=data[int(len(data)*0.9):]

training_data=encode(training_text)
training_data=torch.tensor(training_data, dtype=torch.long)
dev_data=encode(dev_text)
dev_data=torch.tensor(dev_data, dtype=torch.long)

batch_size=32
context_length=8
embedding_size=128
n_head=4
n_layer=4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
vocab_size=len(char_list)
lr= 1e-3



def get_batch(split):
    data=training_data if split=="train" else dev_data
    ix=torch.randint(len(data)-context_length, (batch_size,))
    x=torch.stack([data[i:i+context_length] for i in ix])
    y=torch.stack([data[i+1:i+context_length+1] for i in ix])
    x=x.to(device)
    y=y.to(device)
    return x,y

inp, target=get_batch("train")
print("input:", inp.shape)
print("target:", target.shape)







































input: torch.Size([32, 8])
target: torch.Size([32, 8])


In [4]:
import torch.nn as nn

class Model(nn.Module):

    def __init__(self, vocab_size, embedding_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size)

    def forward(self,x):
        x=self.embedding(x)
        return x


In [5]:
inp,out=get_batch(training_data)
m=Model(vocab_size,embedding_size)
print((m(inp)).shape)


torch.Size([32, 8, 128])


In [6]:
from torch.nn import functional as F
class Attention(nn.Module):

    def __init__(self,head_size,embedding_size):

        super().__init__()

        self.query=nn.Linear(embedding_size,head_size,bias=False)
        self.key=nn.Linear(embedding_size,head_size,bias=False)
        self.value=nn.Linear(embedding_size,head_size,bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(context_length, context_length)))

        self.dropout = nn.Dropout(0.0)

    def forward(self,x):
       
        k=self.key(x)  
        q=self.query(x)
        B,T,H=k.shape
        attention_block=q@k.transpose(-2,-1)*H**-0.5# (B, T, C) @ (B, C, T) -> (B, T, T)
                
        attention_block = attention_block.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        attention_block = F.softmax(attention_block, dim=-1) # (B, T, T)
        attention_block = self.dropout(attention_block)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = attention_block @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out


        





In [7]:
class MultiHeadAttention(nn.Module):

    def __init__(self, head_size, embedding_size):
        super().__init__()
        n_heads=embedding_size//head_size
        self.heads=nn.ModuleList([Attention(head_size,embedding_size) for _ in range(n_heads)])
        self.projection=nn.Linear(embedding_size,embedding_size)
        self.dropout=nn.Dropout(0.0)

    def forward(self,x):
        out=torch.cat([h(x) for h in self.heads],dim=-1)
        out=self.projection(out)
        out=self.dropout(out)
        return out

In [8]:
class MLP(nn.Module):

    def __init__(self,embedding_size):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(embedding_size,4*embedding_size),
            nn.ReLU(),
            nn.Linear(4*embedding_size,embedding_size),
            nn.Dropout(0.0)
        )
    def forward(self,x):
        return self.net(x)
    

In [9]:
class Block(nn.Module):
    def __init__(self,embedding_size,no_head):
        super().__init__()
        self.sa=MultiHeadAttention(embedding_size//no_head,embedding_size)
        self.ffwd=MLP(embedding_size)
        self.ln1=nn.LayerNorm(embedding_size)
        self.ln2=nn.LayerNorm(embedding_size)

    def forward(self,x):
          
        x = self.ln1(x)
        x = x + self.sa(x)
        x = self.ln2(x)
        x = x + self.ffwd(x)
        return x
        


In [10]:
class Model(nn.Module):

    def __init__(self, vocab_size, embedding_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.positional_embedding=nn.Embedding(context_length,embedding_size)
        self.lm_head = nn.Linear(embedding_size, vocab_size)
        self.ln_f = nn.LayerNorm(embedding_size) # final layer norm
        self.blocks = nn.ModuleList([
    Block(embedding_size, n_head)
    for i in range(n_layer)
])
        
    

    def forward(self,x,y=None):
        b,t=x.shape
        pos_emb = self.positional_embedding(
    torch.arange(t, device=x.device)
)
        
        x=self.embedding(x)+pos_emb
        for block in self.blocks:
            x=block(x)

        x=self.ln_f(x)


        logits=self.lm_head(x)


        if y is None:
            loss=None
        else:
            b,t,c=logits.shape
            loss=F.cross_entropy(logits.view(b*t,c),y.view(b*t))
        return logits,loss

            

        

In [11]:
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(100)
        for k in range(100):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [12]:

def train(iterations,lr,model):
    optimizer=optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    for i in range (iterations):
        X,Y=get_batch(training_data)
        logits,loss=model(X,Y)
        if((i+1)%200==0 or i==0):
            losses=estimate_loss(model)
            print(f"step {i+1}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()



        

In [90]:
model1=Model(vocab_size,embedding_size)
model1.to(device)
train(1000,lr,model1)

step 200: train loss 2.4536, val loss 2.2168
step 400: train loss 2.3528, val loss 2.0049
step 600: train loss 2.3265, val loss 1.9569
step 800: train loss 2.2409, val loss 1.8876
step 1000: train loss 2.2675, val loss 1.8839


In [ ]:
model1=Model(vocab_size,embedding_size)
model1.to(device)
train(1000,lr,model1)

#hyperparameters
# batch_size=32
# context_length=8
# embedding_size=128
# n_head=4
# n_layer=4
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# vocab_size=len(char_list)
# lr= 1e-3

step 1: train loss 4.3745, val loss 4.3840
step 200: train loss 2.4144, val loss 2.2176
step 400: train loss 2.3075, val loss 2.0355
step 600: train loss 2.2598, val loss 1.9675
step 800: train loss 2.2446, val loss 1.9573
step 1000: train loss 2.2244, val loss 1.8886


In [14]:
#training with more context length
context_length=32
model1=Model(vocab_size,embedding_size)
model1.to(device)
train(1000,lr,model1)


step 1: train loss 4.3334, val loss 4.3355
step 200: train loss 2.3482, val loss 2.1015
step 400: train loss 2.2076, val loss 1.8432
step 600: train loss 2.1367, val loss 1.7152
step 800: train loss 2.1027, val loss 1.6269
step 1000: train loss 2.0875, val loss 1.5503
